# dl_00_bootstrap

Creates every bronze and control table **empty but correctly typed**,
from the same YAML configs the extractors read.

Why this exists: without it, nothing downstream can run until every
credential is in place and every API has been called successfully. With
it, the whole medallion - silver, gold, the data-quality gate and the
semantic model - can be deployed and verified in real Spark on day one,
with zero data. A schema error then surfaces now rather than at 2am on
the first scheduled run.

Safe to re-run: it only creates tables that do not already exist, so it
never touches loaded data.

In [ ]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/lib")

LIB = "/lakehouse/default/Files"

import yaml
import fabric_common as fc

# Every bronze table has the same shape, because bronze stores the UNPARSED
# payload plus audit columns. That is what makes a transform bug a re-run
# instead of a re-extract.
BRONZE_SCHEMA = (
    "_key string, _project_id string, _merge_key string, _source_endpoint string, "
    "_ingested_at timestamp, payload string, _batch_id string, _row_hash string"
)

CONTROL_TABLES = {
    "dl_meta_watermark": (
        "table_name string, endpoint string, watermark timestamp, "
        "batch_id string, updated_at timestamp"
    ),
    "dl_meta_run_log": (
        "batch_id string, step string, table_name string, row_count long, "
        "status string, message string, logged_at timestamp"
    ),
    "dl_meta_token": "source string, refresh_token string, obtained_at timestamp, batch_id string",
    "dl_dq_results": (
        "batch_id string, expectation string, table_name string, severity string, "
        "failing_rows long, passed boolean, description string, checked_at timestamp"
    ),
    "dl_dq_rejects": (
        "_dq_expectation string, _dq_reason string, _dq_severity string, "
        "_batch_id string, _row string"
    ),
    # The Controller's manual overrides. Landed from Files/reference/ by
    # dl_06_land_reference; declared here so the crosswalk SQL can run before
    # anyone has uploaded a CSV.
    "dl_bronze_reference_project_crosswalk": (
        "procore_project_id string, qbo_customer_id string, hubspot_deal_id string, "
        "reviewed_by string, active boolean"
    ),
}

In [ ]:
def bronze_tables_from_config():
    """Every bronze table named by the three source registries.

    Read from the SAME config the extractors read, so a new endpoint cannot be
    added without its table appearing here too.
    """
    tables = set()

    with open(f"{LIB}/config/procore_endpoints.yml", encoding="utf-8") as handle:
        for entry in (yaml.safe_load(handle) or {}).get("endpoints", []):
            tables.add(entry["bronze_table"])

    with open(f"{LIB}/config/qbo_entities.yml", encoding="utf-8") as handle:
        qbo = yaml.safe_load(handle) or {}
        for entry in qbo.get("entities", []) + qbo.get("reports", []):
            tables.add(entry["bronze_table"])

    with open(f"{LIB}/config/hubspot_objects.yml", encoding="utf-8") as handle:
        hubspot = yaml.safe_load(handle) or {}
        for entry in hubspot.get("objects", []) + hubspot.get("reference", []):
            tables.add(entry["bronze_table"])

    return sorted(tables)


created, existing = [], []
for table in bronze_tables_from_config():
    if spark.catalog.tableExists(table):
        existing.append(table)
        continue
    spark.createDataFrame([], BRONZE_SCHEMA).write.format("delta").saveAsTable(table)
    created.append(table)

for table, schema in CONTROL_TABLES.items():
    if spark.catalog.tableExists(table):
        existing.append(table)
        continue
    spark.createDataFrame([], schema).write.format("delta").saveAsTable(table)
    created.append(table)

print(f"created {len(created)} table(s), {len(existing)} already existed")
for table in created:
    print(f"  + {table}")

In [ ]:
import json
import os


def write_diag(name: str, payload: dict) -> None:
    """Structured diagnostics to Files/_diag/.

    Fabric's job API gives no per-cell detail - a failed notebook reports
    "Failed" and nothing else. Writing what happened to a file the deploy
    scripts can read back is the difference between debugging this and guessing.
    """
    os.makedirs("/lakehouse/default/Files/_diag", exist_ok=True)
    path = f"/lakehouse/default/Files/_diag/{name}.json"
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)
    print(f"diagnostics -> {path}")

write_diag("bootstrap", {"created": created, "existing": existing})
print(f"\n{len(created) + len(existing)} bronze and control tables ready")